# Seeing- and depth-dependent effective source density n_eff: test of the Chang et al. (2013) model on HSC, KiDS and DES conditions

- Author: Sylvie Dagoret-Campagne
- Creation date: 2026-09-21
- Kernel: conda_py313_opsim53 (pure numpy / pandas / matplotlib: no `rubin_sim` needed)
- Context: series `10_DESCMAFDEPTHANDGALCOUNT`. Notebooks 01-03 compare existing `rubin_sim` MAF metrics (depth, galaxy counts, weak-lensing visits) on the dust-footprint variants. None of them contains the seeing. This notebook is the first step towards a **seeing-aware weak-lensing MAF metric**: it builds and tests, on real survey conditions, a forward model of the effective number density of galaxies used for weak lensing, `n_eff`.
- This notebook: **04** of the series. It does not read any OpSim database. It answers: *if the model is given the seeing and depth of HSC, KiDS or DES, how close is its `n_eff` to the published one?*
- Model: Chang et al. 2013, MNRAS 434, 2121 (arXiv:1305.0793v4, i.e. after the 2015 erratum on galaxy sizes).
- Companion notebooks: `01_compareExgalM5withCuts.ipynb`, `02_compareGalaxyCounts.ipynb`, `03_WeakLensing.ipynb`.

## Notebook overview

**The model** (all formulas from Chang et al. 2013 unless stated otherwise).
```
n_eff        = (1/Omega) * sum_i  sigma_SN^2 / (sigma_SN^2 + sigma_m,i^2)       (sigma_SN = 0.26)
sigma_m(nu,R) = a/nu * (1 + (b/R)^c),   (a, b, c) = (1.58, 5.03, 0.39)
nu           = signal-to-noise ratio of the galaxy in its aperture
R            = r_gal^2 / r_PSF^2   (second-moment radii; the seeing enters here and in nu)
selection    : sigma_m < k * sigma_SN                       (k = 1 fiducial)
several visits: sigma_m,joint = (sum_j sigma_m,j^-2)^(-1/2)  (joint fit)  or one coadd (nu_eff, R_eff)
blending     : n_eff -> (1 - F_blend) n_eff,  F_blend = eta * ln(1 + mu * n_raw)
```
`nu` is derived from a 5-sigma point-source depth with the aperture convention of Chang et al. (Appendix A), assuming a background-limited image.

**What is taken from the literature and what is assumed.**

| ingredient | origin |
|---|---|
| `sigma_SN`, `(a, b, c)`, `nu` and `R` definitions, blending law | Chang et al. 2013 (fit done with the KSB/imcat algorithm on PhoSim single exposures) |
| galaxy counts `N(<i) = 46 * 10^(0.31 (i - 25))` arcmin^-2 | LSST Science Book (CFHTLS Deep), quoted valid for 20.5 < i < 25.5 and **extrapolated** to fainter magnitudes here |
| seeing, depth, cuts and published `n_eff` of HSC, KiDS, DES | published papers, see the comments of Section 4 |
| **median size at i = 24, size-magnitude slope, size scatter, galaxy colours, disk-only morphology** | **assumed** (no catalogue is used); they are varied in Section 6 |

**Sections.** 4: configuration of the three surveys. 5: model versus published `n_eff` and ratio `published / model`. 6: sensitivity to the assumed population. 7: dependence on seeing and depth for LSST-like coadds, with the grid saved for later use. 8: LSST-like check against Chang et al. and the DESC SRD, and joint fit versus coadd on a synthetic visit list. 9: how to plug the model into a MAF metric, caveats, references.

**Outputs**: tables and the `n_eff(seeing, depth)` grid in `data_04_NEFF/`, figures in `figs_04_NEFF/`.

## 1. Imports

In [ ]:
import os
from os.path import join

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 2. Configuration

In [ ]:
NB_TAG = "NEFF"
data_dir = f"data_04_{NB_TAG}"
figs_dir = f"figs_04_{NB_TAG}"
os.makedirs(data_dir, exist_ok=True)
os.makedirs(figs_dir, exist_ok=True)


def save_fig(fig, name):
    base = join(figs_dir, name)
    fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
    fig.savefig(base + ".pdf", bbox_inches="tight")
    print("Saved:", base + ".png/.pdf")


# --- model constants (Chang et al. 2013)
SIGMA_SN = 0.26  # intrinsic shape noise per ellipticity component
ABC = (1.58, 5.03, 0.39)  # sigma_m(nu, R) = a/nu * (1 + (b/R)**c), eq. 13
BLEND = {
    1.0: (0.50, 1.9e-3),
    2.0: (0.68, 5.5e-3),
    3.0: (0.62, 1.4e-2),
}  # table 3: deblending radius d [arcsec] -> (eta, mu)
BLEND_D = 2.0  # fiducial deblending radius of Chang et al.

# --- ASSUMED galaxy population (i band); varied in Section 6
POP = dict(
    rh0=0.35,  # median half-light radius [arcsec] of i = 24 galaxies
    xi=0.13,  # d log10(r_h) / d(-m): r_h = rh0 * 10**(-xi * (m_i - 24)), i.e. size ~ flux**0.33
    sig_lnr=0.45,  # lognormal scatter of the size at fixed magnitude
    size_scale=1.0,  # global multiplicative factor on all sizes
)
# --- ASSUMED mean galaxy colours: m_band = m_i + OFFSET[band]
OFFSET = dict(r=0.4, i=0.0, z=-0.1)

## 3. The model

`neff(visits, ...)` takes a list of visits (or of coadds); each entry has its band offset, its 5-sigma point-source depth and its effective seeing FWHM. It integrates over a grid of i-band magnitudes and of log-sizes, so that no galaxy catalogue is needed. Depth and seeing are exactly the two quantities that a cadence simulation gives for every pixel.

In [ ]:
def dndm_i(m):
    # differential counts [arcmin^-2 mag^-1] from N(<i) = 46 * 10**(0.31 * (i - 25))  (LSST Science Book)
    return 46.0 * 0.31 * np.log(10.0) * 10 ** (0.31 * (m - 25.0))


def m5_from_aperture(m_snr, snr, d_ap, fwhm):
    # 5-sigma PSF-weighted point-source depth from a depth quoted at (snr, aperture diameter); background limited
    sig = fwhm / 2.3548
    return m_snr - 2.5 * np.log10((5.0 / snr) * 2.0 * sig / (d_ap / 2.0))


def coadd_m5(m5_visits):
    # coadded 5-sigma depth of visits with individual depths m5_j
    return 1.25 * np.log10(np.sum(10 ** (0.8 * np.asarray(m5_visits, dtype=float))))


def coadd_fwhm(fwhm_visits):
    # effective PSF of the coadd: mean of r_PSF^2 (Chang et al. 2013, section 4.3), r_PSF ~ FWHM
    return np.sqrt(np.mean(np.asarray(fwhm_visits, dtype=float) ** 2))


def sigma_m(nu, R, abc=ABC):
    a, b, c = abc
    return a / nu * (1.0 + (b / R) ** c)


def nu_and_R(m_i, z, visit, pop, f_ap=0.9):
    # nu and R on the (magnitude, size) grid for one visit / coadd
    m_band = m_i + visit["offset"]
    rh = pop["size_scale"] * pop["rh0"] * 10 ** (-pop["xi"] * (m_i - 24.0))  # median half-light radius
    rh = rh[:, None] * np.exp(pop["sig_lnr"] * z[None, :])
    r_gal = 1.46 * rh  # disk: second-moment radius = 1.46 * half-light radius (Chang et al. A4)
    sig = visit["fwhm"] / 2.3548
    r_psf = np.sqrt(2.0) * sig  # second-moment radius of a Gaussian PSF
    r_ap = 1.64 * np.sqrt(r_gal**2 + r_psf**2)  # aperture radius (Chang et al. A10)
    nu_ps = 5.0 * 10 ** (0.4 * (visit["m5"] - m_band))[:, None]  # point-source S/N
    nu = f_ap * nu_ps * 2.0 * sig / r_ap  # 90% of the flux in the aperture, noise ~ aperture radius
    return nu, r_gal**2 / r_psf**2


def neff(
    visits,
    k=1.0,
    mag_lim=None,
    nu_min=None,
    R_min=None,
    sig_m_max=None,
    blend_d=None,
    f_mask=0.0,
    pop=None,
    abc=ABC,
    mrange=(20.0, 28.5),
    dm=0.05,
    nz=41,
):
    # Effective number density [arcmin^-2]. Cuts on mag_lim, nu_min, R_min apply to the first entry of `visits`.
    pop = POP if pop is None else pop
    m_i = np.arange(mrange[0], mrange[1], dm) + dm / 2
    z = np.linspace(-3.5, 3.5, nz)
    wz = np.exp(-0.5 * z**2)
    wz /= wz.sum()
    inv = 0.0
    for j, v in enumerate(visits):
        nu, R = nu_and_R(m_i, z, v, pop)
        inv = inv + 1.0 / sigma_m(nu, R, abc) ** 2
        if j == 0:
            nu0, R0 = nu, R
    sm = 1.0 / np.sqrt(inv)  # joint-fit measurement noise (Chang et al. eq. 16)
    sel = sm < k * SIGMA_SN
    if mag_lim is not None:
        sel &= ((m_i + visits[0]["offset"]) < mag_lim)[:, None]
    if nu_min is not None:
        sel &= nu0 >= nu_min
    if R_min is not None:
        sel &= R0 >= R_min
    if sig_m_max is not None:
        sel &= sm < sig_m_max
    dN = (dndm_i(m_i) * dm)[:, None] * wz[None, :]
    n_raw = float((dN * sel).sum())
    n0 = float((dN * sel * SIGMA_SN**2 / (SIGMA_SN**2 + sm**2)).sum())
    fb = 0.0
    if blend_d is not None:
        eta, mu = BLEND[blend_d]
        fb = eta * np.log(1.0 + mu * n_raw)
    return dict(n_raw=n_raw, n_eff_noblend=n0, blend=fb, n_eff=n0 * (1.0 - fb) * (1.0 - f_mask))

**Sanity checks** on the ingredients (values quoted in the papers are given for comparison).

In [ ]:
print(
    "sigma_m(nu=15, R=1)      = %.3f   (Chang et al. Fig. 4: 0.32 in that bin, fit RMS ~ 0.1)"
    % sigma_m(15.0, 1.0)
)
print("coadd_m5(184 x 24.7)     = %.2f   (LSST 10-year r band: 27.5)" % coadd_m5(np.full(184, 24.7)))
print("coadd_m5(184 x 24.0)     = %.2f   (LSST 10-year i band: 26.8)" % coadd_m5(np.full(184, 24.0)))
m = np.arange(20.0, 25.3, 0.01)
print(
    "N(20 < i < 25.3)         = %.1f arcmin^-2   (LSST Science Book gold sample i < 25.3: ~55)"
    % (dndm_i(m).sum() * 0.01)
)
print(
    "DES i: m5 point source   = %.2f   (from 10-sigma depth 23.8 in a 1.95 arcsec aperture, FWHM 0.88)"
    % m5_from_aperture(23.8, 10.0, 1.95, 0.88)
)

## 4. Surveys: seeing, depth, cuts and published n_eff

Every number below comes from a published source, except the entries marked ASSUMED. The **depth definitions differ from one survey to another** (point-source 5-sigma for HSC and KiDS, 10-sigma in a 1.95 arcsec aperture for DES, converted with `m5_from_aperture`), which is a source of uncertainty of a few tenths of a magnitude.

- **HSC-Y3**: shapes in `i`; mean `i`-band seeing 0.59 arcsec; 5-sigma depth `i ~ 26`; cuts `i < 24.5`, `S/N >= 10`, resolution factor `R2 > 0.3` (i.e. `R = r_gal^2/r_PSF^2 > 0.43`), estimated shape noise < 0.4; published `n_eff = 19.9` arcmin^-2.
- **KiDS-1000**: shapes in `r`; mean seeing 0.7 arcsec; depth `r = 25.2` (limiting magnitude quoted by Chang et al. 2013; its exact definition is not given there, ASSUMED to be a point-source 5-sigma depth); no explicit cut (lensfit weights), so the generic `sigma_m < sigma_SN` cut is used; published `n_eff = 6.17` arcmin^-2.
- **DES-Y3**: shapes from `riz`; DR2 median seeing 0.95 / 0.88 / 0.83 arcsec; DR2 median 10-sigma depth in 1.95 arcsec 24.4 / 23.8 / 23.1; cuts `S/N >= 10` and `T/T_PSF > 0.5` (this size cut is the one of the Y1 catalogue, ASSUMED to be a fair proxy, taken as `R > 0.5`); published `n_eff = 5.59` arcmin^-2 (5.32 with the definition of Chang et al.).

In [ ]:
def make_surveys(offset=OFFSET):
    des = [
        dict(offset=offset["r"], fwhm=0.95, m5=m5_from_aperture(24.4, 10.0, 1.95, 0.95)),
        dict(offset=offset["i"], fwhm=0.88, m5=m5_from_aperture(23.8, 10.0, 1.95, 0.88)),
        dict(offset=offset["z"], fwhm=0.83, m5=m5_from_aperture(23.1, 10.0, 1.95, 0.83)),
    ]
    return {
        "HSC-Y3 (i)": dict(
            visits=[dict(offset=offset["i"], fwhm=0.59, m5=26.0)],
            cuts=dict(mag_lim=24.5, nu_min=10.0, R_min=0.3 / (1.0 - 0.3), sig_m_max=0.4),
            published=19.9,
        ),
        "KiDS-1000 (r)": dict(
            visits=[dict(offset=offset["r"], fwhm=0.70, m5=25.2)],
            cuts=dict(),
            published=6.17,
        ),
        "DES-Y3 (riz)": dict(visits=des, cuts=dict(nu_min=10.0, R_min=0.5), published=5.59),
    }


SURVEYS = make_surveys()
rows = []
for name, sv in SURVEYS.items():
    for v in sv["visits"]:
        rows.append(
            dict(
                survey=name,
                colour_offset=v["offset"],
                fwhm_arcsec=v["fwhm"],
                m5_point_source=round(v["m5"], 2),
            )
        )
display(pd.DataFrame(rows).set_index("survey"))

## 5. Model versus published n_eff

`survey cuts` uses the cuts listed in Section 4; `generic k = 1` uses only `sigma_m < sigma_SN`. `n_raw` is the density of selected galaxies before weighting. The blending term uses `d = 2 arcsec`. The published values are per unmasked area, so no masking factor is applied. `ratio` is `published / model` (survey cuts, with blending): a value of 1 means the model reproduces the published `n_eff`.

In [ ]:
def predict_all(pop=None, offset=OFFSET, blend_d=BLEND_D):
    out = {}
    for name, sv in make_surveys(offset).items():
        out[name] = neff(sv["visits"], blend_d=blend_d, pop=pop, **sv["cuts"])["n_eff"]
    return out


rows = []
for name, sv in SURVEYS.items():
    a = neff(sv["visits"], **sv["cuts"])
    b = neff(sv["visits"], blend_d=BLEND_D, **sv["cuts"])
    g = neff(sv["visits"], blend_d=BLEND_D, k=1.0)
    rows.append(
        {
            "survey": name,
            "published n_eff": sv["published"],
            "model n_raw": a["n_raw"],
            "model n_eff, survey cuts": a["n_eff"],
            "model n_eff, survey cuts + blending": b["n_eff"],
            "model n_eff, generic k=1 + blending": g["n_eff"],
            "ratio published / model": sv["published"] / b["n_eff"],
        }
    )
comparison = pd.DataFrame(rows).set_index("survey")
comparison.to_csv(join(data_dir, "neff_model_vs_published.csv"))
display(comparison.round(2))

## 6. Sensitivity to the assumed galaxy population

The size distribution, the galaxy colours and the deblending radius are not measured here. Each parameter is varied **one at a time** around the nominal values; the envelope of the resulting `n_eff` is an estimate of the modelling uncertainty due to these assumptions (not a full error budget: the algorithm-dependent `(a, b, c)`, the depth definitions and the extrapolated counts are not varied).

In [ ]:
variations = {
    "size scale x0.6": dict(pop=dict(POP, size_scale=0.6)),
    "size scale x1.5": dict(pop=dict(POP, size_scale=1.5)),
    "size slope xi=0.05": dict(pop=dict(POP, xi=0.05)),
    "size slope xi=0.20": dict(pop=dict(POP, xi=0.20)),
    "size scatter 0.30": dict(pop=dict(POP, sig_lnr=0.30)),
    "size scatter 0.60": dict(pop=dict(POP, sig_lnr=0.60)),
    "colours r-i -0.2": dict(offset=dict(OFFSET, r=OFFSET["r"] - 0.2, z=OFFSET["z"] - 0.2)),
    "colours r-i +0.2": dict(offset=dict(OFFSET, r=OFFSET["r"] + 0.2, z=OFFSET["z"] + 0.2)),
    "deblending d=1 arcsec": dict(blend_d=1.0),
    "deblending d=3 arcsec": dict(blend_d=3.0),
}
nominal = predict_all()
table = pd.DataFrame({"nominal": nominal})
for label, kw in variations.items():
    table[label] = pd.Series(predict_all(**kw))
table["min"] = table.drop(columns="nominal").min(axis=1)
table["max"] = table.drop(columns="nominal").max(axis=1)
table["published"] = pd.Series({n: sv["published"] for n, sv in SURVEYS.items()})
table.to_csv(join(data_dir, "neff_sensitivity.csv"))
display(table.round(2))

fig, ax = plt.subplots(figsize=(6.5, 6))
x = table["published"].values
y = table["nominal"].values
ax.errorbar(
    x,
    y,
    yerr=[y - table["min"].values, table["max"].values - y],
    fmt="o",
    capsize=4,
    color="tab:blue",
    label="model (nominal, one-at-a-time envelope)",
)
for name in table.index:
    ax.annotate(
        name,
        (table.loc[name, "published"], table.loc[name, "nominal"]),
        textcoords="offset points",
        xytext=(6, -12),
        fontsize=8,
    )
lim = [3, 30]
ax.plot(lim, lim, "k--", lw=0.8, label="model = published")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_xlabel("published n_eff [arcmin$^{-2}$]")
ax.set_ylabel("model n_eff [arcmin$^{-2}$]")
ax.grid(alpha=0.3, which="both")
ax.legend(fontsize=8)
fig.tight_layout()
save_fig(fig, "neff_model_vs_published")
plt.show()

## 7. Dependence on seeing and depth for LSST-like coadds

Reference coadd: 10-year LSST depths `r = 27.5`, `i = 26.8` (5-sigma, point source) and a coadd effective seeing of 0.8 arcsec (ASSUMED, to be replaced by the seeing of the simulation). Two ways of varying the seeing are shown, because they answer different questions:

- **fixed depth**: only the resolution `R` changes; this isolates the direct effect of the PSF size on the shear measurement noise. Two versions are shown: with the generic cut `sigma_m < sigma_SN` only, and with an additional **resolution cut** `R2 > 0.3` as in HSC (`R2 = R / (1 + R)`, hence `R > 0.43`);
- **fixed exposure**: the depth follows the seeing, `m5 = m5_ref - 2.5 log10(FWHM / 0.8)` (background-limited image); this is what happens for the same total exposure time, and it is the case relevant to a cadence simulation whose per-visit depth already contains the seeing.

The grid `n_eff(seeing, depth shift)` for `r + i` is saved so that a MAF metric can interpolate it instead of integrating over galaxies for every pixel.

In [ ]:
M5_REF = dict(r=27.5, i=26.8)  # 10-year LSST coadd 5-sigma depths
FWHM_REF = 0.8  # ASSUMED reference coadd seeing [arcsec]


def lsst_like(fwhm, dm5=0.0, fixed="depth", bands=("r", "i"), **kw):
    shift = dm5 - (2.5 * np.log10(fwhm / FWHM_REF) if fixed == "exposure" else 0.0)
    visits = [dict(offset=OFFSET[b], fwhm=fwhm, m5=M5_REF[b] + shift) for b in bands]
    return neff(visits, k=1.0, **kw)


seeing = np.arange(0.5, 1.31, 0.1)
fig, axs = plt.subplots(1, 2, figsize=(13, 4.8))
for bands, ls in ((("r", "i"), "-"), (("i",), "--")):
    for fixed, c in (("depth", "tab:blue"), ("exposure", "tab:red")):
        y = [lsst_like(f, fixed=fixed, bands=bands, blend_d=BLEND_D)["n_eff"] for f in seeing]
        label = "+".join(bands) + ", fixed " + fixed
        axs[0].plot(seeing, y, ls, color=c, marker="o", label=label)
y = [lsst_like(f, fixed="depth", blend_d=BLEND_D, R_min=0.3 / 0.7)["n_eff"] for f in seeing]
axs[0].plot(seeing, y, ":", color="tab:green", marker="o", label="r+i, fixed depth + resolution cut R2 > 0.3")
axs[0].set_xlabel("coadd effective seeing FWHM [arcsec]")
axs[0].set_ylabel("n_eff [arcmin$^{-2}$] (blending d = 2 arcsec)")
axs[0].grid(alpha=0.3)
axs[0].legend(fontsize=8)

dm5_axis = np.arange(-1.0, 0.51, 0.25)
grid = np.array([[lsst_like(f, dm5=d, blend_d=BLEND_D)["n_eff"] for f in seeing] for d in dm5_axis])
cs = axs[1].contourf(seeing, dm5_axis, grid, levels=12, cmap="viridis")
fig.colorbar(cs, ax=axs[1], label="n_eff (r + i, fixed depth) [arcmin$^{-2}$]")
axs[1].set_xlabel("coadd effective seeing FWHM [arcsec]")
axs[1].set_ylabel("coadd depth shift w.r.t. r = 27.5, i = 26.8 [mag]")
fig.tight_layout()
save_fig(fig, "neff_vs_seeing_depth_lsst_like")
plt.show()

np.savez(
    join(data_dir, "neff_grid_ri_blend2.npz"),
    seeing=seeing,
    dm5=dm5_axis,
    neff=grid,
    note="n_eff(r+i coadd) [arcmin^-2], blending d=2 arcsec, rows: depth shift, columns: seeing FWHM",
)
print("Saved:", join(data_dir, "neff_grid_ri_blend2.npz"))
for f in (0.6, 0.8, 1.0, 1.2):
    a = lsst_like(f, fixed="depth", blend_d=BLEND_D)["n_eff"]
    b = lsst_like(f, fixed="exposure", blend_d=BLEND_D)["n_eff"]
    c = lsst_like(f, fixed="depth", blend_d=BLEND_D, R_min=0.3 / 0.7)["n_eff"]
    print(
        f"seeing {f:.1f}: n_eff = {a:5.1f} at fixed depth, {c:5.1f} at fixed depth with R2 > 0.3, {b:5.1f} at fixed exposure"
    )

## 8. LSST-like check and joint fit versus coadd

**Reference values**: Chang et al. 2013 (`r + i`, joint fit, `k = 1`): `n_eff = 37` before blending and masking, 31 after blending, 26 after a further 15% masked area. DESC SRD, `Y10` gold sample: `27` arcmin^-2. The model uses the coadd depths of Section 7 and an ASSUMED seeing of 0.8 arcsec, whereas Chang et al. use a distribution of OpSim seeing and sky background, so only an order-of-magnitude agreement is expected.

Then a **synthetic visit list** (ASSUMED: 184 visits per band, single-visit depths `r = 24.7`, `i = 24.0`, lognormal seeing of median 0.8 arcsec and scatter 0.25, depth following the seeing) is used to compare the joint fit over all visits with the single-coadd approximation. Chang et al. find that the coadd gives a `n_eff` about 7% lower than the joint fit.

In [ ]:
a = lsst_like(FWHM_REF)
b = lsst_like(FWHM_REF, blend_d=BLEND_D)
c = lsst_like(FWHM_REF, blend_d=BLEND_D, f_mask=0.15)
ref = pd.DataFrame(
    {
        "model (coadd, seeing 0.8)": [a["n_eff"], b["n_eff"], c["n_eff"]],
        "Chang et al. 2013": [37.0, 31.0, 26.0],
    },
    index=["no blending, no masking", "+ blending (d = 2 arcsec)", "+ 15% masking"],
)
ref["ratio model / Chang"] = ref["model (coadd, seeing 0.8)"] / ref["Chang et al. 2013"]
display(ref.round(2))
print("DESC SRD Y10 gold sample: n_eff = 27 arcmin^-2")

rng = np.random.default_rng(42)
n_vis = 184
single = dict(r=24.7, i=24.0)  # 30 s single-visit 5-sigma depths
visits, coadds = [], []
for band in ("r", "i"):
    fwhm = np.clip(FWHM_REF * np.exp(0.25 * rng.standard_normal(n_vis)), 0.5, 1.8)
    m5 = single[band] - 2.5 * np.log10(fwhm / FWHM_REF)
    visits += [dict(offset=OFFSET[band], fwhm=f, m5=m) for f, m in zip(fwhm, m5)]
    coadds.append(dict(offset=OFFSET[band], fwhm=coadd_fwhm(fwhm), m5=coadd_m5(m5)))
    m5c, fwc = coadds[-1]["m5"], coadds[-1]["fwhm"]
    print(f"band {band}: coadd depth {m5c:.2f}, coadd effective seeing {fwc:.3f} arcsec")

joint = neff(visits, k=1.0, blend_d=BLEND_D)
coad = neff(coadds, k=1.0, blend_d=BLEND_D)
nj, nc = joint["n_eff"], coad["n_eff"]
print(
    f"joint fit over {len(visits)} visits: n_eff = {nj:.1f}, coadd: n_eff = {nc:.1f}, coadd / joint = {nc / nj:.3f}"
)

## 9. Towards a seeing-aware MAF metric

What this notebook provides for the next step:

1. `neff(visits, ...)`: one call per pixel from the per-visit `fiveSigmaDepth` and effective seeing of an OpSim database (band offsets from `OFFSET`). It is slow for many visits; for a full-sky map, interpolate the saved grid `neff_grid_ri_blend2.npz` in (coadd effective seeing, coadd depth) after computing `coadd_m5` and `coadd_fwhm` per pixel.
2. The depth maps of notebook 01 (`ExgalM5WithCuts`) already give a dust- and coverage-cut coadd depth per pixel; a per-pixel coadd effective seeing has to be computed in addition (mean of `FWHM^2` over the visits).
3. Total statistical weight of a footprint: `N_eff = sum_pixels n_eff * pixel_area`; the noise term of the shear power spectrum is `sigma_SN^2 / n_eff` (Chang et al. 2013, eq. 11).

## Caveats

- `(a, b, c)` come from the KSB algorithm on single exposures with the true PSF. Metacalibration, lensfit, re-Gaussianization and other estimators have other noise properties; the ratios `published / model` of Section 5 measure this and the other modelling errors together, and they differ between surveys. They should not be applied as one universal factor.
- The galaxy counts are extrapolated beyond i = 25.5, and the size distribution, colours and disk-only morphology are assumptions (Section 6 varies them).
- At fixed depth, the seeing dependence of the model is weak with the generic cut, because the fitted `sigma_m(nu, R)` depends weakly on `R` (exponent 0.39); it becomes strong as soon as an explicit resolution cut is applied (Section 7). The seeing dependence of the real estimators is therefore set largely by the resolution criteria of the pipeline, which is not constrained by the three published `n_eff` alone.
- `nu` is computed in a background-limited approximation from a 5-sigma point-source depth; depth definitions differ between surveys.
- Photometric redshift selection, tomographic binning, masking (published `n_eff` are per unmasked area), intrinsic alignments and PSF-modelling systematics are **not** in this model. The multiplicative and additive PSF biases are a separate term (see the propagation of PSF errors of Paulin-Henriksson et al. 2008).
- In a cadence simulation, the relative comparison between runs is expected to be more robust than the absolute `n_eff`.

## References
- Chang, C. et al. 2013, MNRAS 434, 2121, arXiv:1305.0793 (erratum: MNRAS 447, 1746, 2015)
- LSST Science Collaboration 2009, LSST Science Book, arXiv:0912.0201
- LSST DESC 2018, Science Requirements Document, arXiv:1809.01669
- Heymans, C. et al. 2012, MNRAS 427, 146 (definition of the weighted `n_eff`)
- Li, X. et al. 2022 (HSC-Y3 shape catalogue); Giblin, B. et al. 2021 (KiDS-1000 shear catalogue); Gatti, M. et al. 2021 (DES Y3 shape catalogue); DES Collaboration 2021, ApJS 255, 20 (DES DR2)
- Paulin-Henriksson, S. et al. 2008, A&A 484, 67; Massey, R. et al. 2013, MNRAS 429, 661 (PSF requirements)
- Lochner, M. et al. 2021, arXiv:2104.05676 (MAF weak-lensing proxy and 3x2pt emulator)